# 第二章：收益、风险与一份可信的研究报告

第一章的账户告诉我们持有什么，以及钱怎样流动。这一章接着问：**同样从 1000 元出发，一个股票、债券和现金组合的结果，应当怎样比较、解释和承受？** 最终产物是一张能追溯回逐日账本的风险报告，而不是一个单独的夏普比率。

本章始终围绕一个最简实现：每天期初把账户的 60% 配到股票、30% 配到债券，其余留现金，交易从漂移后的实际持仓出发，并扣除费用。我们先用 12 天教学收益逐项手算，再用本地 S&P 500 快照检查真实市场中的波动聚集和回撤。这个配置只是让收益、相关性、换手和成本同时出现的教学载体，不预设它优于其他配置。

## 0. 学习目标

读完后，你应当能从账户金额计算简单收益和复利财富，说明对数收益的用途与限制，区别资产收益、组合收益与主动收益；能从同一账本得到成本、漂移权重、回撤、波动、因子暴露和尾部风险；能解释年化和风险比率依赖的假设，并把一个“高收益”拆成市场暴露、杠杆、现金、费用和研究误差。

读者已经掌握概率统计，因此我们不重新推导均值方差和基础分布。重点是：这些工具用在金融账户时，分母是什么、时间对应什么、哪些统计假设会失效，以及一个数值究竟支持多强的研究结论。

## 1. 先固定收益的对象、期间和分母

假设一个没有外部存取款的账户从 1000 元变成 1012 元，本期简单收益是 \((1012-1000)/1000=1.2\%\)。这里的分子是整段时间内账户财富变化，分母是期初自有财富，既不是某项资产的价格，也不是保证金占用。

令期末总财富为 \(V_t\)，一期简单收益为

\[
r_t=\frac{V_t-V_{t-1}}{V_{t-1}},\qquad V_t=V_{t-1}(1+r_t).
\]

\(t-1\) 和 \(t\) 是前后两个估值时点，\(V\) 包括现金、证券市值和已确认的权利负债；若只用价格计算，就需要额外处理股息、拆股和其他现金流。一个价格指数的收益不是自动包含分红的总回报，第一章的这一区别贯穿本章。

若投资者中途存入 100 元，账户从 1000 到 1100，不代表赚了 10%。策略研究通常采用剔除外部现金流后的**时间加权收益（time-weighted return）**：在存取款处划分子期间，再把各段收益连乘。投资者实际资金经历则可以用**资金加权收益（money-weighted return）**，即根据投入、取出和期末金额求内部收益率。两者分别回答管理过程表现和这笔资金经历；现金流发生在期初、期末还是期间，需要明确。

### 1.1 为什么组合相加用简单收益

如果期初用 600 元持有股票、300 元持有债券、100 元留现金，这三部分一期分别回报 1%、-0.2%、0.01%，期末盈利为 \(600\times1\%+300\times(-0.2\%)+100\times0.01\%=5.41\) 元，账户收益 0.541%。这里权重是期初价值占比，资产之间用同一期简单收益加权。

**对数收益（log return）** 定义为 \(g_t=\log(V_t/V_{t-1})=\log(1+r_t)\)。它在同一财富路径的时间方向上可以相加，\(\sum_tg_t=\log(V_T/V_0)\)，但不能直接拿各资产对数收益按权重相加来精确表示组合对数收益。精确做法是先计算 \(r_{p,t}=\sum_iw_{i,t-1}r_{i,t}\)，再取 \(\log(1+r_{p,t})\)。\(w_{i,t-1}\) 是该期开始前已确定的资产权重。

一个很清楚的极端例子是价格从 100 跌到 50，又回到 100。简单收益为 -50% 和 +100%，相加得到 +50%，但实际没赚钱；连乘得到 \(0.5\times2=1\)。对数收益是 \(\log0.5\) 和 \(\log2\)，相加为零。它们没有矛盾，只是可加的方向不同。对数还要求财富为正，负财富或破产边界不能继续套用。




![100 到 50 再回到 100 的往返路径上两种收益口径的累加差异](figures/ch02_simple_vs_log.png)

**图 2-1｜simple 与 log 的可加性差异。** 价格从 100 到 50 再回 100，财富没有变化。简单收益直接相加却为 50%，对数收益相加为 0。多期财富应使用简单收益连乘，或先累加对数收益再取指数，不能把两种公式混用。数据类型：三点合成路径。


In [ ]:
import numpy as np
import pandas as pd

# 教学收益为各交易区间内已经处理现金分配的资产总回报。
# 不对应真实股票、债券或一份可直接交易的数据产品。
dates = pd.bdate_range("2024-02-01", periods=12)
asset_returns = pd.DataFrame({
    "equity": [-.04, .025, .015, -.03, -.02, .01, .035, -.01, .005, -.025, .02, .01],
    "bond": [.003, -.002, .001, .002, -.004, .001, -.003, .002, 0, -.001, .002, .001],
}, index=dates)
cash_return = pd.Series(0.0001, index=dates, name="cash_return")
# 每行都是该行收益发生前的目标权重；剩余 10% 为现金。
target = pd.DataFrame({"equity": 0.6, "bond": 0.3}, index=dates)
gross_before_cost = (target * asset_returns).sum(axis=1) + 0.1 * cash_return
pd.concat([asset_returns, gross_before_cost.rename("portfolio_gross")], axis=1)


第一天股票跌 4%、债券涨 0.3%，不计费用的账户收益为
\(0.6(-4\%)+0.3(0.3\%)+0.1(0.01\%)=-2.309\%\)。债券的上涨缓冲了股票损失，但没有让账户免于亏损。代码中的每一行收益对应一个完整持有区间；目标权重在区间开始前决定，不能读完该行收益再决定该行权重。

我们没有声称可以无误差地按任何真实市场收盘价成交。教学收益只提供一个精确定义的区间；应用到真实数据时，需要将它替换成实际成交之后到下次估值的回报。若信号用了当天收盘信息，而持有收益也从昨天收盘开始，就会把信号发生前已经经历的损益算给它。

## 2. 复利：每天做对一点，和最后赚到多少是两回事

把上述逐期收益还原成财富路径，需要连乘：

\[
V_T=V_0\prod_{t=1}^T(1+r_t).
\]

\(V_0\) 是初始资本，\(T\) 是已观察的期数，\(r_t\) 必须是同一账户口径下、包含应计费用的简单收益。一天 +10%、一天 -10%，100 元最后只剩 \(100\times1.1\times0.9=99\) 元。算术平均收益为零，却不表示复利增长为零。

**几何平均收益（geometric mean return）** 是 \((V_T/V_0)^{1/T}-1\)，表示每期固定赚多少才能得到同一终值；算术平均则描述样本单期收益的平均值。前者回答历史路径增长，后者进入单期预期和均值方差模型。它们有不同用途，不应互相替代。

小收益下 \(\log(1+r)\approx r-r^2/2\)，所以平均对数增长约为算术均值减去与二阶矩有关的项，常进一步近似为 \(\mu-\sigma^2/2\)。这称为**波动拖累（volatility drag）**：在可比算术均值下，波动较大的乘法路径通常增长较慢。它不是一笔额外扣掉的交易费用，也不意味着任何低波动资产都优于高波动资产；预期收益、分布尾部和配置约束仍然重要。




![相同平均日收益、不同波动的两条合成财富路径](figures/ch02_vol_drag.png)

**图 2-2｜波动拖累：均值相同，终点不同。** 两组日收益被调整为相同的样本算术均值，高波动组的复利终值仍更低。小收益近似下，平均对数收益约为 \(\bar r-s^2/2\)；两条路径的预期对数增长差约为 \((s_H^2-s_L^2)/2\)。下图使用收益率横轴展示两组分布。数据类型：固定 seed 合成样本，单次路径不代表一般保证。


In [ ]:
comparison = pd.DataFrame({
    "account_return": gross_before_cost,
    "wealth": 1000 * (1 + gross_before_cost).cumprod(),
    "cumulative_log_return": np.log1p(gross_before_cost).cumsum(),
})
pd.Series({
    "period_arithmetic_mean": gross_before_cost.mean(),
    "period_geometric_mean": np.expm1(np.log1p(gross_before_cost).mean()),
    "12_period_total_return": (1 + gross_before_cost).prod() - 1,
    "final_wealth": comparison.wealth.iloc[-1],
})


### 2.1 年化是尺度转换，不能制造样本

如果 12 个观察期是交易日，约定一年有 \(N=252\) 个这样的交易期，那么已实现**复合年化增长率（compound annual growth rate，CAGR）** 是

\[
R_{ann}=\left(V_T/V_0\right)^{N/T}-1.
\]

\(N\) 是一年采用的交易期数，\(T\) 是同口径下的观察期数。这是在问“过去这段乘法增长若按同样平均速度持续一年，数值是多少”，不是在预测未来。12 天赚一点就年化成很大数字，不会增加证据强度；一年 252 日也不适用于所有频率和交易日历。全天候市场、月度策略和跨交易所组合需要自己的期间定义。

算术均值乘 252 是“年化算术均值”的惯用近似，不等于这条历史路径的 CAGR。若想估计预期财富增长，又需额外考虑未来收益分布、相关性和可重复性。本章对短教学样本主要报告区间收益，只有进入较长真实样本时，才把年化指标作为便于比较的摘要。

## 3. 加入实际交易：从漂移持仓出发计算换手

每天恢复 60% 股票、30% 债券，不等于每天都没有交易。第一天股票跌得更多，期末股票占比就低于 60%；第二天恢复目标，需要补买股票，同时相应调整债券和现金。这种持有后比例改变称为**权重漂移（weight drift）**。

**再平衡（rebalancing）** 是从当前持仓调回目标。**换手（turnover）** 衡量交易规模，本章定义为本期各证券买卖金额绝对值之和除以交易前账户价值：

\[
TO_t=\sum_i\left|w^{target}_{i,t-1}-w^{drift}_{i,t-1}\right|.
\]

\(w^{target}\) 是本期目标证券金额占交易前净值的比例，\(w^{drift}\) 是上一期结束后真实持仓金额占当前净值的比例；求和只覆盖被交易的股票、债券，不把现金流转再计一次。某些报告会将买卖金额和的一半定义为换手，也有只报买入额的口径，成本计算必须配套。一个已经持有 60/30 目标、第二天价格完全没动的账户，换手接近零；从全现金开始建仓的账户，第一天换手为 0.9。

令每单位买卖金额费用率为 \(c\)，则费用金额是 \(cTO_tV_{t-1}\)。我们将目标证券金额按交易前财富确定，再从剩余现金里扣费，所以证券目标仍是 60% 和 30%，现金略少于 10%。这是一项明确的自融资约定；如果所有资产目标加起来恰好 100%，就不能又全额买入又从不存在的现金扣费，需要缩小目标或明确融资。

下面的实现直接维护金额，不通过“今天目标减昨天目标”偷算换手。它包含首次建仓费用，且现金实际减少后才计息。函数仅支持非负权重、非负现金的现货配置；多空融资、期货保证金和税费还需要另外建账。


In [ ]:
def run_account(returns, target_weights, cash_rates, cost_rate=0.0005, initial_value=1000.0):
    """目标证券金额按期初净值确定，费用由现金支付。"""
    if not returns.index.equals(target_weights.index) or not returns.index.equals(cash_rates.index):
        raise ValueError("收益、目标和现金利率的时间索引必须完全一致")
    if list(returns.columns) != list(target_weights.columns):
        raise ValueError("资产列及顺序必须一致")
    if returns.isna().any().any() or target_weights.isna().any().any() or cash_rates.isna().any():
        raise ValueError("请先明确处理缺失数据，不把缺失收益当零")
    if (returns <= -1).any().any() or (target_weights < 0).any().any() or cost_rate < 0:
        raise ValueError("本例只支持正价值现货资产、非负权重和非负费用")
    if (target_weights.sum(axis=1) > 1).any():
        raise ValueError("目标持仓不能超过净值")
    holdings = np.zeros(returns.shape[1])
    value = float(initial_value)
    records = []
    for day in returns.index:
        weights = target_weights.loc[day].to_numpy(dtype=float)
        asset_r = returns.loc[day].to_numpy(dtype=float)
        rf = float(cash_rates.loc[day])
        desired = value * weights
        turnover = float(np.abs(desired - holdings).sum() / value)
        fee = cost_rate * turnover * value
        cash = value - desired.sum() - fee
        if cash < -1e-10:
            raise ValueError("现金不足以支付本期交易费用")
        equity_pnl, bond_pnl = desired * asset_r
        cash_pnl = cash * rf
        holdings = desired * (1 + asset_r)
        new_value = holdings.sum() + cash + cash_pnl
        net = new_value / value - 1
        gross = float(weights @ asset_r + (1 - weights.sum()) * rf)
        records.append({"date": day, "value": new_value, "gross_return": gross,
                        "net_return": net, "turnover": turnover, "fee": fee,
                        "equity_pnl": equity_pnl, "bond_pnl": bond_pnl,
                        "cash_pnl": cash_pnl, "equity_weight_after": holdings[0] / new_value,
                        "bond_weight_after": holdings[1] / new_value})
        value = new_value
    return pd.DataFrame(records).set_index("date")

account = run_account(asset_returns, target, cash_return)
account[["value", "gross_return", "net_return", "turnover", "fee"]].round(6)


第一天从全现金开始，买股票 600、债券 300，换手 0.9，费用率 5 个基点产生 \(900\times0.0005=0.45\) 元费用。余下现金为 99.55，收盘股票 576、债券 300.9，现金计息后为 99.559955，总价值为 976.459955。对应净收益约 -2.3540%。费用使现金可赚的利息也略微减少，因此净收益与毛收益之差是 \(cTO_t(1+r_{f,t})\)，而非完全等于 \(cTO_t\)。本例影响极小，但金额账本让约定保持一致。

第二天的股票初始权重是 \(576/976.459955\)，不是昨天设定的 60%；债券也是用其真实期末金额作分子。即使两天目标完全一样，也需要交易。把换手写成 `target.diff()` 会将首次建仓和每天漂移后的调整都漏掉，产生看似低成本的虚假结果。

实际费用通常不止一个比例。佣金和交易税随市场规则发生；穿越报价价差需要支付相对中间价的差；市场冲击随订单规模、波动和流动性变化，不能保证与交易额线性。这里的固定 \(c\) 是可解释的第一层近似。研究应看费用提高后结论是否仍存在，而不是把 5 个基点当成放之四海皆准的真实数字。

## 4. 波动：同样的平均结果，途中有多不稳定

从账本拿到 `net_return` 后，样本标准差描述单期财富变化的离散程度。我们不再重复它的统计定义，而要注意对象：股票波动是股票收益的波动，账户波动还受权重、现金和交易费用影响。降低所有风险资产权重会降低波动，却也通常同时降低预期收益；不能单凭低波动就称之为更好的信号。

金融里常见**年化波动（annualized volatility）** \(\hat\sigma_{ann}=s_{daily}\sqrt{252}\)。\(s_{daily}\) 是日收益样本标准差，252 是约定的一年交易日数。它来自对长期间收益相加时的方差缩放近似，不是复利财富公式，更不是所有市场的自然定律。

若收益有自相关，\(h\) 期相加收益的方差为

\[
\operatorname{Var}\!\left(\sum_{j=1}^{h}r_j\right)
=h\gamma_0+2\sum_{k=1}^{h-1}(h-k)\gamma_k.
\]

\(\gamma_0\) 是单期方差，\(\gamma_k\) 是相隔 \(k\) 期收益的自协方差，\(h\) 是聚合期数。只有交叉项可以忽略时，才得到平方根时间缩放。正自相关、重叠持有收益和不活跃资产的估值平滑都会改变长期风险；日波动很小的产品，也可能只是价格没有及时更新。

### 4.1 风险是随时间变化的

**波动聚集（volatility clustering）** 指剧烈变化往往相邻出现，而平静期也会持续。条件波动变化并不自动意味着收益本身有强可预测均值；它首先告诉我们风险状态在变化。下面的真实图用 60 日滚动窗口，危机期与平静期明显不同。一个全样本年化数会将它们混在一起。




![真实 S&P 500 日收益与 60 日滚动年化波动](figures/ch02_vol_clustering.png)

**图 2-3｜波动是聚集的。** 上半部分是真实日收益的正负茎线，下半部分是 60 日滚动年化波动：2011、2018、2020、2022 等危机期波动成簇出现，平静期可以长期低于中位数。因此“一个年化波动数字”只是窗口平均，报告必须写明估计窗口和频率。数据类型：公开数据本地快照（Yahoo Finance ^GSPC，2010 年起）；仅用于教学。


回到 12 天账户，前几天的大幅股票变动会带来较高组合波动；如果真实策略按风险预算调整仓位，只能用调仓前已经知道的数据估计波动。假设今天结束后计算 3 日股票波动，最早应作用于下个可交易区间，因此需要显式移位。滚动窗口开始时没有足够数据也不能凭空补出估计；应预先规定保持基准仓位或等待建仓。


In [ ]:
known_vol = asset_returns.equity.rolling(3).std().shift(1)
# 教学规则：已知日波动超过 2.5% 时，股票从 60% 降为 30%，其余留现金。
# 阈值事先写定，仅展示信息时间，不从这 12 天搜索最优值。
defensive_target = target.copy()
defensive_target["equity"] = np.where(known_vol > 0.025, 0.3, 0.6)
defensive_account = run_account(asset_returns, defensive_target, cash_return)
pd.DataFrame({"known_vol_before_period": known_vol,
              "equity_target": defensive_target.equity,
              "baseline_net": account.net_return,
              "defensive_net": defensive_account.net_return}).round(5)


这仍是同一本账户：只改变事前目标，漂移换手、费用和风险评估共用同一实现。防守规则可能减小某次损失，也可能在反弹前减仓而错过恢复；估计窗口、阈值和交易延迟都会影响它。12 天的结果只能帮助理解机制，不能证明“波动控制有效”。真正的研究要在没有参与选择规则的时间段比较基准与防守配置，并同时观察成本和错失上涨。

## 5. 分散化：债券为什么能缓冲，又为什么可能失灵

股票和债券不是因为名称不同就自动分散风险。组合波动取决于它们在同一持有期间是否共同变化。对固定的事前权重 \(w\) 和资产收益协方差矩阵 \(\Sigma\)，组合方差为 \(w^\top\Sigma w\)。\(\Sigma\) 的对角元是各资产方差，非对角元是共同变化项，现金近似固定的微小收益在这里暂不计波动。

本章股票权重 0.6、债券权重 0.3，因此

\[
\sigma_p^2=0.6^2\sigma_E^2+0.3^2\sigma_B^2
+2(0.6)(0.3)\rho_{EB}\sigma_E\sigma_B.
\]

\(\sigma_E,\sigma_B\) 是股票、债券同频收益波动，\(\rho_{EB}\) 是相关系数。若年化波动分别为 20% 和 10%，相关为零，则组合波动约 \(\sqrt{0.36(0.2)^2+0.09(0.1)^2}=12.37\%\)；若相关为 1，则变为 15%。同样两种资产、同样仓位，仅共同变化关系不同，风险就不同。




![两资产收益散点与不同相关系数下组合波动随权重的变化](figures/ch02_correlation_diversification.png)

**图 2-4｜相关性如何决定分散化。** 左图为相关系数约 0.6 的合成二元样本。右图按两资产组合方差公式计算：其他条件相同，相关系数 \(\rho\) 越低，组合的最低波动越低；\(\rho=1\) 且权重非负时，组合波动才等于两资产波动的加权平均。数据类型：固定 seed 样本与公式计算。


In [ ]:
weights = np.array([0.6, 0.3])
sample_cov = asset_returns.cov()
formula_vol = float(np.sqrt(weights @ sample_cov.to_numpy() @ weights))
direct_vol = float((asset_returns @ weights).std(ddof=1))
pd.Series({"portfolio_vol_from_covariance": formula_vol,
           "portfolio_vol_from_weighted_returns": direct_vol,
           "equity_bond_sample_correlation": asset_returns.corr().loc["equity", "bond"]})


两种波动计算一致，是因为它们描述同一个固定权重、未扣费用的线性组合，而不是因为某种估计方法更高级。实际 `account.net_return` 还包含随漂移变化的费用，风险控制版本的权重也随时间改变，就不能把一个全样本固定 \(w\) 公式直接当成完整账户风险。

### 5.1 市值权重和风险贡献不是一回事

我们的股票占 60%、债券占 30%，股票风险却可能占绝大部分，因为股票自身波动更大。对给定协方差矩阵，组合波动对权重的导数为 \((\Sigma w)_i/\sigma_p\)，第 \(i\) 项的**风险贡献（risk contribution）** 为 \(RC_i=w_i(\Sigma w)_i/\sigma_p\)，它们相加等于 \(\sigma_p\)。这个量说明某项配置为整体波动贡献多少，不是它预期赚多少钱。

以相关为零、股票波动 20%、债券波动 10% 的例子，股票方差项 0.0144，债券方差项 0.0009，股票贡献约占总方差的 94%。看起来接近“两类资产平衡”的市值配置，风险上可能仍高度偏向股票。若某资产有效对冲其他资产，它的风险贡献也可能为负，不能把贡献比例机械解释成独立风险概率。

相关性不是合同常数。通胀冲击可能使股票因估值受压下跌，同时债券因利率上升下跌；平时的负相关便不成立。把估计的相关系数放进优化器，会把样本噪声放大成极端仓位，尤其在资产多、样本短时。**收缩估计（shrinkage）** 把不稳定样本协方差向一个更稳定的结构拉近；因子模型则用少数共同风险解释资产共变。它们减少估计自由度，但仍需要检查状态变化和压力情景，不能消除未来未知。

## 6. 暴露与杠杆：低净仓位不等于低风险

我们的主线是只有多头的现货组合，股票和债券总权重为 0.9，净权重也为 0.9，现金约为 0.1。对一般多空配置，**总暴露（gross exposure）** 是 \(\sum_i|w_i|\)，**净暴露（net exposure）** 是 \(\sum_iw_i\)。权重仍用名义持仓金额除以自有资本，负权重表示空头。

例如用 1000 元资本做多股票组合 1000 元，同时卖空另一组 1000 元，总暴露为 2，净暴露为 0。它可能对市场方向相对中性，但并没有消失的风险：两组股票可能差异扩大，空头可能难以借入或被召回，卖出所得现金可能被冻结为担保，多空同时平仓也需要成本。不能因为净值是零就把整笔交易当作没有资本需求。

**杠杆（leverage）** 的含义应随合同说明。普通现货常用总名义规模除以资本；债券还需看久期，期权还需看 Delta、Gamma 和其他敏感度，期货还要看乘数和保证金。一张 1000 元名义规模的期货与 1000 元权利金的期权不是等风险头寸。

**因子暴露（factor exposure）** 把问题从持有多少元推进到受什么风险影响。以超额收益作市场回归：

\[
r_{p,t}-r_{f,t}=\alpha+\beta(r_{b,t}-r_{f,t})+\varepsilon_t.
\]

\(r_p\) 是账户收益，\(r_f\) 是同频现金基准，\(r_b\) 是市场基准，\(\beta\) 是市场变化的线性敏感度，\(\alpha\) 是该模型在该样本中没有由市场项解释的平均部分，\(\varepsilon\) 是剩余波动。对于本章组合，股票持仓自然带来正市场 beta；不能因为组合赚钱就说获得了独立超额收益。

截距还可能吸收遗漏的债券风险、错误的基准、非线性或数据偏差。它是特定模型下的统计量，只有交易成本、可见时间、样本外表现和其他风险解释都经得起检查时，才可能接近研究者关心的可实现 alpha。下一章会进一步讨论这些模型与因果解释的距离。

## 7. 回撤：账户从高点走到了哪里

波动关注逐期变化，**回撤（drawdown）** 关注财富相对历史最高点的缺口。将初始财富也放进路径，令

\[
M_t=\max_{0\le u\le t}V_u,\qquad D_t=V_t/M_t-1.
\]

\(M_t\) 是截至当时的财富峰值，\(D_t\le0\) 是当时回撤。**最大回撤（maximum drawdown）** 在本章按负数报告，为所有 \(D_t\) 的最小值；若报告正的损失幅度，应写明符号。初始时点 \(u=0\) 很重要：我们第一天就从 1000 下跌，若从第一天期末财富才开始累计峰值，这次损失会被漏掉。

本章第一天回撤就是约 -2.3540%。即使第二天有正收益，只要财富仍低于 1000，账户仍处于回撤中。**回撤持续时间（drawdown duration）** 是从峰值开始到重新达到该峰值的经过时间；**恢复时间（time to recovery）** 也常用于描述从谷底回到前高的时间，不同报告定义会有差异，必须注明起点。若样本结束尚未恢复，要记作“未恢复”，不能把最后一天当成恢复日。

下面的真实图把财富曲线和历史峰值一起画，再用水下曲线单独显示缺口。短促暴跌与缓慢下跌可能有相似深度，却有不同的资金压力和等待体验。




![真实 S&P 500 财富曲线与水下曲线](figures/ch02_drawdown_real.png)

**图 2-5｜最大回撤与恢复时间。** 上半部分是再基准化的财富曲线与历史峰值线，下半部分是水下曲线（drawdown）。2020-02-19 到 2020-03-23 的价格回撤约 -33.9%，于 2020-08-18 恢复前高；2022 年的回撤更慢、更磨人。读图时同时记录深度、持续时间和恢复时间，三者缺一不可。数据类型：公开数据本地快照（Yahoo Finance ^GSPC）；仅用于教学。


In [ ]:
def wealth_path(returns, initial_value=1000.0):
    """显式保留初始财富，避免漏掉第一期回撤。"""
    values = np.r_[initial_value, initial_value * (1 + np.asarray(returns)).cumprod()]
    peaks = np.maximum.accumulate(values)
    return pd.DataFrame({"wealth": values, "peak": peaks, "drawdown": values / peaks - 1})

path = wealth_path(account.net_return)
path.index.name = "period_number_0_is_initial"
path.round(5)


回撤依赖样本长度和路径。一个只运行三个月的策略最大回撤小于十年策略，并不自动表示它更安全；它可能尚未经历相关市场状态。同一组日收益重新排列，终值不变，最大回撤和恢复时间却可能改变。因此回撤不能只靠一个正态波动估计推出来，也不能作为精确的未来亏损上界。

还要考虑**流动性风险（liquidity risk）**：账面仍有 950 元，不代表能在需要时全部变现成 950 元现金。若急售导致报价扩大、市场冲击上升，实际退出净值会更低。对于杠杆账户，某次中途回撤可能触发保证金或风险限额，令它根本无法等到模型中的后续恢复。这是逐日现金约束比只看最终收益更重要的原因。

## 8. 风险调整收益：先问和什么相比

**夏普比率（Sharpe ratio）** 用超出同频现金基准的平均收益除以其标准差：

\[
S=\frac{\overline{r_p-r_f}}{s(r_p-r_f)}.
\]

\(r_p-r_f\) 是每期超额收益，分子是该序列的样本均值，分母是该序列的样本标准差。按常见弱相关近似，日频比率乘 \(\sqrt{252}\) 作年化。现金收益率必须先转成与账户相同期间的收益；将年利率 4% 每天直接减掉，是单位错误。

夏普比率回答每单位一般波动对应多少平均超额收益，不回答最大可能亏损。策略如果平时收取少量收益、少数日遭遇巨大损失，短样本夏普可能很好看；估值不及时造成收益平滑，也会压低分母。调很多参数挑出最大样本夏普，又会把选择噪声伪装成能力。

**索提诺比率（Sortino ratio）** 用低于某目标收益的下行偏差作分母。若目标为 \(\tau\)，一种常见口径是 \(DD=\sqrt{T^{-1}\sum_t\min(r_t-\tau,0)^2}\)，比率为 \((\bar r-\tau)/DD\)。\(\tau\) 是每期最低目标，\(T\) 是所有观察期数；这里包含零下行项，而不是只对亏损日求标准差。它更关注不想要的方向，但目标选择和尾部样本不足仍会影响结果。

**信息比率（information ratio，IR）** 则把现金基准替换成投资基准：\(IR=\overline{r_p-r_b}/s(r_p-r_b)\)。主动收益 \(r_p-r_b\) 的标准差称为**跟踪误差（tracking error）**。本章 60/30/10 组合相对全股票可能在下跌时更好、上涨时更差；这首先来自风险配置不同，不等于选股或择时能力。研究目标若是检验防守规则，最直接的基准应是同样记账、同样成本的固定配置版本。


In [ ]:
excess = account.net_return - cash_return
active_vs_equity = account.net_return - asset_returns.equity
floor = 0.0
downside_deviation = np.sqrt(np.minimum(account.net_return - floor, 0).pow(2).mean())
# 12 天只演示同频定义，不将这些短样本比率宣称为稳定年化能力。
ratio_demo = pd.Series({
    "period_sharpe": excess.mean() / excess.std(ddof=1),
    "period_sortino_target_zero": (account.net_return.mean() - floor) / downside_deviation,
    "period_information_ratio_vs_equity": active_vs_equity.mean() / active_vs_equity.std(ddof=1),
})
ratio_demo


比较两种规则时，收益与风险还应对齐资金约束。若防守规则因为平均只投了 40% 股票而波动更低，可以先和相近风险配置的基准比较，再讨论是否额外改善了择时。单纯把原组合杠杆加大，可能提高收益和回撤，而不提高同口径夏普；一旦融资费用、尾部和限额介入，连这种简单缩放也不成立。

## 9. 尾部风险：一个分位点以外还有多坏

将单期损失定义为 \(L_t=-r_t\)，于是亏损为正。**风险价值（Value at Risk，VaR）** 是给定持有期、置信水平下的损失分位点。若日度 95% VaR 为 2%，表示在所用分布模型下，约 5% 的日损失会更大；它没有说最坏只亏 2%，也没有说明超过后平均亏多少。

**预期损失（expected shortfall，ES）**，也常称条件风险价值（CVaR），关注最差尾部的平均损失。对连续分布，95% ES 可写为超过 95% VaR 后的平均损失；有离散样本或分位点大量并列时，需要处理尾部概率边界，不能不加说明地将所有“低于某分位点”的日子一股脑平均。




![真实日收益直方图上的 5% VaR 与 CVaR](figures/ch02_var_cvar.png)

**图 2-6｜VaR 与 ES 的符号口径。** 横轴为日简单收益。灰色线画在 5% 收益分位点 \(q_{0.05}<0\)，图中报告的损失风险取正值：\(\mathrm{VaR}_{0.95}=-q_{0.05}\)。ES 是最差 5% 样本日平均收益的相反数；它与 VaR 都按损失为正的口径报告。数据类型：公开 S&P 500 本地快照，历史样本不承诺未来风险。


以损失由小到大的样本估计 VaR，就是**历史模拟法（historical simulation）**；它容易解释，却默认这段历史足以代表未来风险。我们的 12 天样本中，最差 5% 只有 0.6 个观察日。此时任何“95% 风险估计”都主要由一个最差日控制，没有精确的尾部证据。拉长样本能增加尾部观察，但又会混入不同市场状态。

下面用经验分布的分位点定义 VaR，并按最差尾部所占的概率质量计算 ES：若需要 1.5 个样本的尾部，就取最差一个完整权重加次差一个半权重。对于 12 天的 5% 尾部，它会只由最差日决定。代码可以算出一个数，但解释它为什么脆弱，比保留更多小数位有用。


In [ ]:
def historical_tail_risk(returns, confidence=0.95):
    losses = -np.asarray(returns, dtype=float)
    if len(losses) == 0 or not 0 < confidence < 1:
        raise ValueError("需要非空收益与 0 到 1 之间的置信水平")
    var = float(np.quantile(losses, confidence, method="inverted_cdf"))
    worst_first = np.sort(losses)[::-1]
    mass = (1 - confidence) * len(losses)
    boundary_weights = np.clip(mass - np.arange(len(losses)), 0, 1)
    es = float((worst_first * boundary_weights).sum() / mass)
    return pd.Series({"VaR_loss_positive": var, "ES_loss_positive": es,
                      "tail_effective_observations": mass})

historical_tail_risk(account.net_return)


### 9.1 压力测试把经济情景接回账本

**压力测试（stress test）** 不要求先给坏事分配精确概率。我们直接设股票 -10%、债券 -5%、现金利息不变，计算当前配置会怎样。按 60/30/10 权重，资产损失约 \(0.6(-10\%)+0.3(-5\%)=-7.5\%\)。这检查的是“股票和债券一起跌”的失效情形，不能由平常低相关关系替代。

下一层可以设报价价差扩大三倍、现金无法及时划拨、只能成交一半订单，再重新跑持仓和现金。对于债券，可以先设收益率上升 100 个基点、信用利差扩大 150 个基点，再按久期或完整现金流重估；对于期权，应同时改变标的、波动率和剩余期限。情景必须对应合同真实敏感度，而不是给所有价格列统一减 10%。

我们也要区分当前已持有组合受到冲击的损益与“冲击前恰好全现金，再建仓”的损益。后者会多计算一次建仓费用，且可能假设了并不存在的成交机会。下面第一列只算当前配置承受的价格冲击；需要模拟退出时，再另外加入退出交易和流动性成本。


In [ ]:
stress = pd.DataFrame({
    "equity": [-0.10, -0.10, 0.08],
    "bond": [0.02, -0.05, -0.03],
}, index=["equity_crash_bond_hedge", "stocks_and_bonds_down", "equity_up_bond_down"])
stress["portfolio_before_exit_cost"] = stress[["equity", "bond"]] @ weights
stress


## 10. 收益从哪里来：让每一元都能解释

**损益归因（PnL attribution）** 把总财富变化拆成来源。本章账户每期有精确恒等式：股票持仓损益加债券持仓损益，加实际剩余现金利息，再减交易费用，等于账户财富变化。它比先看一个汇总比率更有诊断力：股票在赚钱，组合仍亏钱，可能是债券下跌或交易太频繁；如果全部收益都来自股票上涨，就不要把结果描述成一个独立的预测信号。

这个金额归因可以跨期直接相加；收益率贡献若要跨期汇总则需考虑财富基数变化。把每天股票收益贡献简单相加，再称为其全年复合贡献，通常不等于终值归因。我们的代码保留的是每期实际金额，因此相加后应恰好等于期末减期初财富。


In [ ]:
attribution = pd.Series({
    "equity_pnl": account.equity_pnl.sum(),
    "bond_pnl": account.bond_pnl.sum(),
    "cash_interest": account.cash_pnl.sum(),
    "trading_fees": -account.fee.sum(),
})
attribution["sum_of_components"] = attribution.sum()
attribution["actual_wealth_change"] = account.value.iloc[-1] - 1000
attribution.round(6)


### 10.1 合理的基准先于更复杂的模型

若研究问题是“每天再平衡是否值得”，可与买入后不再调整的同起始配置比较；若研究问题是“已知波动上升后减仓是否有用”，就与本章固定 60/30/10 配置比较，保留相同数据、费率和资金约束。全股票基准有助于展示方向风险，却不完全回答相同风险预算下的规则价值。

**基准（benchmark）** 不是随便找一条更容易打败的曲线。它应尽可能保留研究者原本就能获得的简单收益来源，只去掉正在检验的新增机制。复杂机器学习信号则应先面对相同执行条件下的简单规则，而不是只和全现金比较。

下面同时报告固定配置、风险控制版本和全股票价格收益。第三列仅为资产收益基准，没有假装已经建成与前两者相同现金约束的交易账户；若正式比较可投资表现，还要补上它的建仓、交易费和资金规则。名字里把这个区别写出来，可以避免汇总表遮蔽对象差异。


In [ ]:
def compact_report(returns):
    r = pd.Series(returns, dtype=float)
    p = wealth_path(r, initial_value=1.0)
    return pd.Series({"periods": len(r), "total_return": p.wealth.iloc[-1] - 1,
                      "period_volatility": r.std(ddof=1), "max_drawdown": p.drawdown.min(),
                      "worst_period": r.min()})

reports = pd.DataFrame({
    "fixed_allocation_net": compact_report(account.net_return),
    "defensive_rule_net": compact_report(defensive_account.net_return),
    "equity_asset_before_implementation_cost": compact_report(asset_returns.equity),
})
reports.round(5)


### 10.2 统计上能分辨，和经济上值得做

一个平均每天改善 0.5 个基点的信号，即使在大样本中能与零区分，也可能被 3 个基点交易成本吃掉；一个低频策略效应看起来很大，却只有十次决策，估计范围可能极宽。研究报告应同时呈现收益大小、采样不确定性与实现成本。

本章只用 12 天，我们可以验证记账恒等式，却无法可靠估计长期均值、极端风险和危机相关性。若要估计置信区间，普通独立重抽样会破坏时间相关和波动聚集；保留一段连续时间的**区块重抽样（block bootstrap）** 更符合某些依赖结构，但区块长度与状态稳定性仍需判断。下一章会把这种金融时间依赖作为计量问题展开。

还有研究选择本身的不确定性。若试了 100 个窗口、阈值和资产，最后一个漂亮结果与只做一次预先规定的实验不同。应记录候选范围、保留未参与选择的时期、检查费用升高和参数小幅变化后的结果，并关注这些证据是否共同支持一个清楚的经济机制。增加一个小数点位不能替代这些步骤。

## 11. 本地真实实验：从价格指数到完整路径报告

真实市场这一段仍沿用“收益—财富—风险”的最简实现，但不冒充含分红基金账户。我们从本地 S&P 500 价格指数收盘快照计算每日简单收益；不加入不存在的股息，不把指数点位当成真实成交基金价格。这个实验适合观察风险状态和路径，不能直接用于证明 ETF 策略的可实现回报。

先读取股票自己的交易日历并计算收益，再选定研究窗口。没有因为别的市场多开市一天就插入虚假零收益，也没有把缺失价格前向填充后宣称风险下降。滚动 60 日波动只用到当日结束时的信息，图上它是事后描述；若用于下一期仓位，仍需加交易延迟。


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    found = [p for p in [base, base / "research", base / "run-quant" / "research"]
             if (p / "sources/external/data/yahoo_sp500.csv").exists()]
    if found:
        research_root = found[0]
        break
else:
    raise FileNotFoundError("未找到 research/sources/external/data 本地快照")

raw = pd.read_csv(research_root / "sources/external/data/yahoo_sp500.csv", parse_dates=["date"])
close = raw.set_index("date")["close"].sort_index().dropna()
real_returns = close.pct_change(fill_method=None).loc["2010":"2025"].dropna()
real_wealth = (1 + real_returns).cumprod()
# 将窗口初始财富 1 纳入峰值；不能漏掉第一天相对窗口起点的亏损。
real_peaks = real_wealth.cummax().clip(lower=1.0)
real_drawdown = real_wealth / real_peaks - 1
rolling_vol = real_returns.rolling(60).std(ddof=1) * np.sqrt(252)
real_excess_assuming_zero_cash_rate = real_returns  # 只为单独展示假设，不当成实际历史现金收益。
real_summary = compact_report(real_returns)
real_summary["annualized_geometric_price_return"] = np.expm1(np.log1p(real_returns).mean() * 252)
real_summary["annualized_volatility_convention"] = real_returns.std(ddof=1) * np.sqrt(252)
real_summary["annualized_arithmetic_mean"] = real_returns.mean() * 252
real_summary


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
real_wealth.plot(ax=axes[0], label="price wealth", color="#2563eb")
real_peaks.plot(ax=axes[0], label="running peak", color="#64748b", linestyle="--")
axes[0].legend()
axes[0].set_ylabel("rebased wealth")
rolling_vol.plot(ax=axes[1], color="#d97706")
axes[1].set_ylabel("60-day annualized vol")
axes[2].fill_between(real_drawdown.index, real_drawdown, 0, color="#dc2626", alpha=.4)
axes[2].set_ylabel("drawdown")
fig.tight_layout()
plt.show()

annual_groups = real_returns.groupby(real_returns.index.year)
yearly_report = pd.DataFrame({
    "price_return": annual_groups.apply(lambda r: (1 + r).prod() - 1),
    "annualized_volatility": annual_groups.std(ddof=1) * np.sqrt(252),
    "worst_day": annual_groups.min(),
    "observations": annual_groups.size(),
})
yearly_report.loc[2018:2023].round(4)


对照财富与波动图，找一个波动已经回落、财富却还没恢复前高的区间。这说明“市场恢复平静”和“账户已经回本”是两件事。再比较 2020 与 2022，思考快跌快修复与较长回撤给资金安排带来的不同要求。逐年表也提醒我们：一个全样本平均包含了很多不同的经历。

这里没有减去历史无风险利率，所以不将均值除波动冒充真实历史现金超额夏普。若要计算它，需加入当时可获得的现金工具收益、匹配计息日期和持有区间，再讨论税费、借贷利差和可投资性。缺少一列必要数据时，保留清楚的限制，比凭空补一个“常见年利率”更可维护。

## 12. 常见误解

**“平均收益相同，最后就一样。”** 本章连乘路径已经展示算术平均和几何增长不同。应先说明比较的是单期期望、已实现终值还是相同风险下的复利增长，不能把它们合并成一句“收益更高”。

**“固定目标权重没有换手。”** 持有期间价格变化使真实权重漂移，下一次恢复目标就发生交易。`target.diff()` 为零，只说明目标没变，不说明订单没发生。

**“最大回撤是未来亏损上限。”** 它只是这段样本中相对历史峰值经历的最深缺口；更长持有期、新状态和退出冲击都可能带来更差结果。还必须把初始财富纳入峰值，避免漏算首日损失。

**“相关性低就不用压力测试。”** 相关性是历史条件下的统计摘要。共同通胀、融资紧缩和被迫抛售可能令原有对冲同时失效，情景分析必须允许这种情况发生。

**“净仓位为零，风险也为零。”** 多空差异、借券、融资、集中度和尾部仍然存在。总暴露、因子暴露与现金约束需要一起报告。

**“夏普高就可以实盘。”** 成本漏算、估值平滑、短样本和多次选择都可能抬高比率。更可信的依据是可解释机制、同口径基准、未参与选择的样本，以及逐笔能追溯的钱和风险。

## 13. 手工练习

### 练习 A：首日自融资账本

从 1000 元全现金开始，目标股票 60%、债券 30%，现金 10%；股票首期 -4%、债券 +0.3%、现金 +0.01%；按买卖金额收 5 个基点费用。先求毛收益，再写出交易后现金、期末三部分金额、账户净收益。若直接从第一天期末开始累计历史峰值，会漏掉什么？

<details><summary>参考答案</summary>

毛收益为 -2.309%。买入股票 600、债券 300，交易 900，费用 0.45，现金变为 99.55。期末股票 576、债券 300.9、现金 99.559955，总财富 976.459955，净收益约 -2.3540045%。初始峰值应为 1000；若第一天才建立峰值，会把首日回撤错误记成零。

</details>

### 练习 B：组合不是两只波动的平均

股票年化波动 20%，债券 10%，权重分别 60%、30%，现金 10% 视为无波动。相关系数分别为 1、0、-1 时，计算组合年化波动。相关为 -1 是否意味着此权重正好完全对冲？

<details><summary>参考答案</summary>

相关为 1 时波动为 \(0.6(20\%)+0.3(10\%)=15\%\)；相关为零时约 12.37%；相关为 -1 时为 \(|0.6(20\%)-0.3(10\%)|=9\%\)。此权重没有完全对冲；完全抵消要求两部分带符号风险规模相等，且完全负相关关系在未来持续成立。

</details>

### 练习 C：回撤与恢复

初始财富 100，连续四期收益为 -10%、+10%、-5%、+20%。写出财富、历史峰值、各期回撤和最大回撤，并说明何时首次恢复初始高点。

<details><summary>参考答案</summary>

财富依次为 90、99、94.05、112.86；连同初始 100，峰值依次为 100、100、100、112.86。回撤依次为 -10%、-1%、-5.95%、0，最大回撤为 -10%。第四期首次超过初始高点，谷底在第一期，谷底到恢复经历三个期间；峰值到恢复经历四个期间。应说明报告采用哪一种持续时间。

</details>

### 练习 D：成本和基点

某研究报告年化毛收益约 12%，证券买卖金额之和除以净值定义的年换手为 20。每单位交易金额成本依次为 5、20、50 个基点，估算净收益，并说明为什么正式结果应重新跑账本。

<details><summary>参考答案</summary>

成本近似为 1%、4%、10%，净收益约 11%、8%、2%。这是用平均规模和线性费用做的粗估；逐日财富、漂移换手、现金利息、买卖价格和冲击会改变实际复合结果。正式研究应按每期现金支出重算，不把年末减一个数当成精确净值。

</details>

### 练习 E：同样赚钱，哪一个是新能力

固定配置赚 8%，某防守版本赚 7%，全股票赚 12%。仅凭这三项是否能判断防守规则失败？至少还需要什么，才能回答“它是否值得”？

<details><summary>参考答案</summary>

不能。需要同口径净收益、波动、回撤和恢复、交易成本、风险敞口、样本区间与不确定性。若防守版本显著减轻可承受的坏情形，或在相近风险预算下改善表现，可能有价值；也可能只是长期少持股票而没有额外择时能力，应与简单降仓基准比较。选择目标应在观察结果前规定。

</details>

## 14. 自测与思考

1. 简单收益与对数收益各在哪个方向可加，组合对数收益应该怎样计算？
2. 为什么 12 天可以验证账户恒等式，却不足以可靠估计 95% ES 和长期夏普？
3. 如果一个策略的样本回归截距为正，为什么还不能宣布有可交易 alpha？
4. 如果所有已实现历史收益只是重新排序，哪些指标不变，哪些可能变化？
5. 本章将费用从剩余现金扣除。如果目标证券权重改成 100%，应该先改什么？

<details><summary>参考答案</summary>

1. 同期简单收益按期初资金权重加总；单条正财富路径的对数收益跨期相加。组合应先算简单收益，再取 `log1p`，不能直接加权各资产对数收益当作精确结果。
2. 记账恒等式是已定义合同和算法必须满足的代数关系；长期风险和收益是未知分布特征。12 天的 5% 尾部不到一个样本，均值、相关性和危机行为都高度不确定。
3. 它依赖基准、遗漏因子、时间对齐和模型形式，还可能受到数据选择、成本漏计与多次试验影响，需要合理经济解释与未参与选择的验证。
4. 同一组净收益的复利终值、样本均值和标准差不变；最大回撤、恢复时间和时间相关可变。若重新排序的是资产收益再重新执行带状态和成本的策略，连净收益集合也可能变化，不能继续套用这个结论。
5. 必须留出费用资金、按扣费后净值缩小目标，或显式允许并计价融资。本例函数会拒绝现金不足，不能为了让代码继续跑就忽略缺口。

</details>

## 15. 本章小结

同一份 60/30/10 账本把本章串了起来：期初金额决定简单收益，连乘决定财富，漂移后的持仓决定真实交易，费用和现金利息决定可实现净值。协方差解释一般共同风险，因子暴露解释受到哪些市场变化影响，回撤解释从高点到谷底的经历，VaR、ES 和压力测试检查坏情形的不同侧面。

一份可用的研究报告需要同时呈现对象、期间、毛净收益、基准、暴露、成本、财富路径、风险状态和估计不确定性。指标的价值在于帮助解释同一个经济实验，不能靠增加术语替代证据。下一章会把视角推进到计量识别、金融时序和模型验证，继续回答哪些关系可以预测、哪些解释有因果含义、哪些发现能在新数据中留下来。
